In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv('../assets/patient/patient_concat.csv', index_col=0)
df

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
import joblib

tfidf_vectorizer = TfidfVectorizer(lowercase=True)
X_full = df['message'].sort_index().values
y_full = df['message_type'].sort_index().values
X_train, X_test, y_train, y_test = train_test_split(df['message'], df['message_type'], test_size=0.2, random_state=42)
message_tfidf = tfidf_vectorizer.fit_transform(X_train)
joblib.dump(tfidf_vectorizer, '../assets/nn_vectorizer.joblib')

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.utils import to_categorical
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
y_train_encoded = le.fit_transform(y_train)
y_train_categorical = to_categorical(y_train_encoded)
y_test_encoded = le.fit_transform(y_test)
y_test_categorical = to_categorical(y_test_encoded)

x_test = tfidf_vectorizer.transform(X_test)

model = Sequential()

model.add(Dense(units=50, activation='relu', input_shape=(message_tfidf.shape[1],)))

model.add(Dense(units=30, activation='relu'))

model.add(Dense(units=20, activation='relu'))

model.add(Dense(len(le.classes_), activation='softmax'))

In [ ]:

model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy','categorical_crossentropy' ,'mean_squared_error', 'mean_absolute_error'])

with tf.device('/gpu:0'):
    model.fit(message_tfidf.toarray(), y_train_categorical, epochs=30, validation_data=(x_test.toarray(), y_test_categorical)) 

model.save('../assets/neural_network.h5')
joblib.dump(le, '../assets/label_encoder.joblib')